In [1]:
!pip install -q groq pandas

In [2]:
!pip install groq

In [3]:
import pandas as pd
from groq import Groq

In [6]:
import os
from google.colab import userdata
gem_key = userdata.get('groq')
os.environ['GROQ_API_KEY'] = gem_key

In [7]:
client = Groq()
client

In [8]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Hello"
        }
    ]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [9]:
reviews = [
    "The product arrived on time and works perfectly.",
    "Excellent quality and value for money.",
    "Product is okay but packaging was damaged.",
    "Average performance, nothing special.",
    "Terrible quality and stopped working after one day.",
    "Waste of money and poor customer service.",
    "Good design but battery drains quickly.",
    "Fast delivery and amazing build quality.",
    "Not bad, but expected better.",
    "Very satisfied with the purchase."
]

In [14]:
df = pd.DataFrame(
    reviews,
    columns=["review"]
)

df

,review
0,The product arrived on time and works perfectly.
1,Excellent quality and value for money.
2,Product is okay but packaging was damaged.
3,"Average performance, nothing special."
4,Terrible quality and stopped working after one...
5,Waste of money and poor customer service.
6,Good design but battery drains quickly.
7,Fast delivery and amazing build quality.
8,"Not bad, but expected better."
9,Very satisfied with the purchase.


# # FEW-SHOT CHAIN-OF-THOUGHT PROMPT

In [31]:
prompt_template = """
You are an expert sentiment analyst.

Analyze the customer review carefully.

Follow these steps:

Step 1: Identify all positive-sentiment phrases.

Step 2: Identify all negative-sentiment phrases.

Step 3: Check for contradictions or mixed sentiment.

Step 4: Explain the reasoning step-by-step.

Step 5: Decide the final sentiment label.

Sentiment Labels:

Positive:
- Mostly positive opinions.
- Customer is satisfied.
- Minor negatives can be ignored if positives dominate.

Neutral:
- Mixed positive and negative opinions with similar strength.
- Neither clearly positive nor clearly negative.
- Factual or balanced review.

Negative:
- Mostly negative opinions.
- Customer is dissatisfied.
- Minor positives can be ignored if negatives dominate.

------------------------------------------------

Example 1

Review:
"The phone is excellent and battery lasts long."

Step 1: Positive-sentiment phrases
- excellent
- battery lasts long

Step 2: Negative-sentiment phrases
- none

Step 3: Contradictions or mixed sentiment
- no

Step 4: Reasoning
The review contains only strong positive feedback.
No negative comments are present.
The customer is satisfied with the product.

Step 5: Final Label
Positive

------------------------------------------------

Example 2

Review:
"The product is okay but packaging was damaged."

Step 1: Positive-sentiment phrases
- okay

Step 2: Negative-sentiment phrases
- packaging was damaged

Step 3: Contradictions or mixed sentiment
- yes

Step 4: Reasoning
The review contains both positive and negative feedback.
The positive and negative opinions are balanced.
Therefore the sentiment is mixed.

Step 5: Final Label
Neutral

------------------------------------------------

Example 3

Review:
"Terrible quality and stopped working after one day."

Step 1: Positive-sentiment phrases
- none

Step 2: Negative-sentiment phrases
- terrible quality
- stopped working after one day

Step 3: Contradictions or mixed sentiment
- no

Step 4: Reasoning
The review contains strong negative feedback.
The customer is clearly dissatisfied.

Step 5: Final Label
Negative

------------------------------------------------

Now analyze this review:

Review:
{review}

Return output exactly in this format:

Step 1: Positive-sentiment phrases
...

Step 2: Negative-sentiment phrases
...

Step 3: Contradictions or mixed sentiment
...

Step 4: Reasoning
...

Step 5: Final Label
Positive / Neutral / Negative
"""

# # FUNCTION

In [47]:
def analyze_review(review):

    prompt = prompt_template.format(
        review=review
    )

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

# # TEST SINGLE REVIEW

In [33]:
review = "Good design but battery drains quickly."

result = analyze_review(review)

print(result)

Step 1: Positive-sentiment phrases
- Good design

Step 2: Negative-sentiment phrases
- battery drains quickly

Step 3: Contradictions or mixed sentiment
- Yes. The review contains both a positive comment about the design and a negative comment about the battery life, indicating mixed sentiment.

Step 4: Reasoning
The reviewer praises the product’s design, which is a clear positive statement. However, they also complain that the battery drains quickly, which is a clear negative statement. Both points are expressed with similar strength and no one outweighs the other. Because the review contains both positive and negative sentiments of comparable intensity, it reflects a balanced or mixed viewpoint rather than a clear leaning toward satisfaction or dissatisfaction.

Step 5: Final Label
Neutral


 # ANALYZE ALL REVIEWS

In [42]:
sentiments = []

for output in results:

    text = output.lower()

    if "step 5: final label" in text:

        final_part = text.split("step 5: final label")[-1]

        if "positive" in final_part:
            sentiments.append("Positive")

        elif "negative" in final_part:
            sentiments.append("Negative")

        elif "neutral" in final_part:
            sentiments.append("Neutral")

        else:
            sentiments.append("Unknown")

    else:
        sentiments.append("Unknown")

In [43]:
df["sentiment"] = sentiments

df[["review", "sentiment"]]

,review,sentiment
0,The product arrived on time and works perfectly.,Positive
1,Excellent quality and value for money.,Positive
2,Product is okay but packaging was damaged.,Neutral
3,"Average performance, nothing special.",Neutral
4,Terrible quality and stopped working after one day.,Negative
5,Waste of money and poor customer service.,Negative
6,Good design but battery drains quickly.,Neutral
7,Fast delivery and amazing build quality.,Positive
8,"Not bad, but expected better.",Neutral
9,Very satisfied with the purchase.,Positive
